# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. You will load the metadata and records, examine record sets and field structure by `@id`, and perform exploratory analysis and simple visualizations.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
# Print basic metadata summary
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Temporal Coverage: {metadata.temporalCoverage}")

# List top-level record sets if available
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print("\nAvailable record sets:")
    for rs in metadata.recordSet:
        print(f"- {rs['@id']}")
else:
    print("\nNo record sets defined at the top-level metadata. We'll list them in the next section.")

## 2. Data Overview

Review available record sets, fields, and their IDs from the Croissant schema.

In Croissant, each `RecordSet`, `Field`, and `Column` is uniquely identified by its `@id`. Let's enumerate record sets and the structure of the dataset.

In [ ]:
# List all record sets with their @ids and fields
record_set_ids = []
print("\n=== Record Sets Overview ===\n")
for record_set in dataset.record_sets:
    print(f"Record Set @id: {record_set['@id']}")
    record_set_ids.append(record_set['@id'])
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            fid = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
            print(f"    - {fid}")
    else:
        print("  No fields defined.")
    print()
if not record_set_ids:
    print("No record sets discovered in schema.")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Be sure to reference record sets and fields by their `@id` values.

**Below we'll:**
    
1. List all record set `@id`s found.
    
2. Load all records of each set into separate pandas DataFrames for exploration.

In [ ]:
# Extract data for each record set by @id
dataframes = {}
available_record_sets = record_set_ids  # From previous cell

if not available_record_sets:
    print("No record sets found for data extraction.")
else:
    for rs_id in available_record_sets:
        print(f"\nLoading records from record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records. Columns:")
        print(df.columns.tolist())

    # For demonstration, select the first available record set for further analysis
    if available_record_sets:
        main_rs_id = available_record_sets[0]
        print(f"\nUsing '{main_rs_id}' for downstream analysis.")
        display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps with `mlcroissant`-derived DataFrames:
- Filtering records by field value
- Normalizing numeric fields
- Grouping and summary statistics

Below, pick a numeric field `@id` and a group (category) field `@id` shown in previous steps.

In [ ]:
# For demonstration, identify likely numeric and grouping fields from the DataFrame
df = dataframes[main_rs_id]
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numeric fields: {numeric_fields}")
print(f"Group fields: {group_fields}")

# Choose a numeric field and group field (@id), fallback if not found
numeric_field = numeric_fields[0] if numeric_fields else None
group_field = None
# Try to pick a likely categorical field other than the numeric
for candidate in group_fields:
    if candidate != numeric_field:
        group_field = candidate
        break

if numeric_field:
    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouped summary
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped by '{group_field}', mean of '{numeric_field}':")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization

Visualize the distribution of a numeric field or explore relationships between fields using matplotlib.

In [ ]:
if numeric_field:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if available
    if group_field and group_field in df.columns and df[group_field].nunique()<20:
        plt.figure(figsize=(10,5))
        df.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load a Croissant-structured dataset using `mlcroissant`, inspect its structure by `@id`, extract record sets as DataFrames, and perform basic filtering, normalization, and visualizations.

For detailed field/column semantics, always consult the record set and field `@id` documentation in the dataset schema for precise data interpretation.